In [1]:
import sys

In [2]:
from collections import Counter
from pathlib import Path
import json

dataset_path = Path("../data/finch_dataset.json")
records = json.loads(dataset_path.read_text(encoding="utf-8"))

financial = [
    row for row in records
    if row["db_id"] == "financial" and row["db_name"] == "bird"
]

ids = [row["question_id"] for row in financial]

print("Total FINCH records:", len(records))
print("BIRD financial records:", len(financial))
print("Partition counts:", dict(Counter(row["partition"] for row in financial)))
print("Difficulty counts:", dict(Counter(row["difficulty"] for row in financial)))
print("Question ID range:", min(ids), "to", max(ids))
print("Unique question IDs:", len(set(ids)))
print("Known orientation IDs present:", sorted(set(ids) & {1, 2, 3}))

Total FINCH records: 75725
BIRD financial records: 106
Partition counts: {'dev': 106}
Difficulty counts: {'medium': 37, 'easy': 62, 'hard': 7}
Question ID range: 1 to 106
Unique question IDs: 106
Known orientation IDs present: [1, 2, 3]


In [3]:
from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import json
import sqlite3

dataset_path = Path("../data/finch_dataset.json").resolve()
db_path = Path("../data/financial.sqlite").resolve()

records = json.loads(dataset_path.read_text(encoding="utf-8"))
financial = [
    row for row in records
    if row["db_id"] == "financial" and row["db_name"] == "bird"
]

ORIENTATION_IDS = {1, 2, 3}
SELECTION_SALT = "domain-adapt-financial-v1"

split_spec = {
    "working_dev": {"easy": 6, "medium": 4, "hard": 2},
    "frozen_eval": {"easy": 12, "medium": 9, "hard": 3},
}

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def selection_rank(row: dict) -> str:
    value = (
        f"{SELECTION_SALT}:"
        f"{row['difficulty']}:"
        f"{row['question_id']}"
    )
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

eligible = [
    row for row in financial
    if row["question_id"] not in ORIENTATION_IDS
]

by_difficulty = defaultdict(list)
for row in eligible:
    by_difficulty[row["difficulty"]].append(row)

for difficulty in by_difficulty:
    by_difficulty[difficulty].sort(key=selection_rank)

selected = {"working_dev": [], "frozen_eval": []}
offsets = Counter()

for role, difficulty_counts in split_spec.items():
    for difficulty, count in difficulty_counts.items():
        start = offsets[difficulty]
        stop = start + count
        selected[role].extend(by_difficulty[difficulty][start:stop])
        offsets[difficulty] = stop

print("Dataset SHA-256:", file_sha256(dataset_path))
print("Database SHA-256:", file_sha256(db_path))

for role, rows in selected.items():
    print(f"\n{role}:")
    print("  total:", len(rows))
    print("  difficulty:", dict(Counter(r["difficulty"] for r in rows)))
    print("  IDs:", sorted(r["question_id"] for r in rows))

Dataset SHA-256: c1e462743e4891fecf0fc10dbc6b0eb554fac56a186bdeedec9ccd662e0a4130
Database SHA-256: d15d89cdb068a202b6f2b99342af44dffc1d52545b39ceaf62efdc0ba570101e

working_dev:
  total: 12
  difficulty: {'easy': 6, 'medium': 4, 'hard': 2}
  IDs: [5, 11, 14, 16, 45, 49, 61, 69, 81, 89, 99, 105]

frozen_eval:
  total: 24
  difficulty: {'easy': 12, 'medium': 9, 'hard': 3}
  IDs: [4, 6, 10, 12, 13, 17, 28, 29, 30, 35, 37, 38, 39, 41, 47, 52, 53, 58, 62, 80, 82, 93, 96, 104]


In [4]:
def audit_reference(row: dict, conn: sqlite3.Connection) -> dict:
    try:
        cursor = conn.execute(row["SQL"])
        result = cursor.fetchall()
        column_count = len(cursor.description or [])

        concerns = []

        if len(result) == 0:
            concerns.append("empty result")

        if any(value is None for result_row in result for value in result_row):
            concerns.append("contains NULL")

        if (
            len(result) == 1
            and len(result[0]) == 1
            and isinstance(result[0][0], (int, float))
            and result[0][0] == 0
        ):
            concerns.append("single numeric zero")

        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": True,
            "row_count": len(result),
            "column_count": column_count,
            "concerns": concerns,
        }

    except sqlite3.Error as error:
        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": False,
            "row_count": None,
            "column_count": None,
            "concerns": [f"{type(error).__name__}: {error}"],
        }

conn = sqlite3.connect(f"file:{db_path.as_posix()}?mode=ro", uri=True)

for role, rows in selected.items():
    audits = [audit_reference(row, conn) for row in rows]
    flagged = [
        audit
        for audit in audits
        if not audit["executes"] or audit["concerns"]
    ]

    print(f"\n{role}:")
    print("  executed:", sum(a["executes"] for a in audits), "/", len(audits))
    print("  flagged:", flagged)


working_dev:
  executed: 12 / 12
  flagged: [{'question_id': 99, 'difficulty': 'easy', 'executes': True, 'row_count': 1, 'column_count': 1, 'concerns': ['single numeric zero']}, {'question_id': 14, 'difficulty': 'easy', 'executes': True, 'row_count': 0, 'column_count': 1, 'concerns': ['empty result']}, {'question_id': 49, 'difficulty': 'medium', 'executes': True, 'row_count': 1, 'column_count': 1, 'concerns': ['single numeric zero']}, {'question_id': 5, 'difficulty': 'medium', 'executes': True, 'row_count': 1, 'column_count': 1, 'concerns': ['single numeric zero']}, {'question_id': 11, 'difficulty': 'medium', 'executes': True, 'row_count': 0, 'column_count': 1, 'concerns': ['empty result']}, {'question_id': 105, 'difficulty': 'medium', 'executes': True, 'row_count': 0, 'column_count': 3, 'concerns': ['empty result']}, {'question_id': 81, 'difficulty': 'hard', 'executes': True, 'row_count': 1, 'column_count': 1, 'concerns': ['contains NULL']}]

frozen_eval:
  executed: 24 / 24
  flagge

In [5]:
working_audits = {
    row["question_id"]: audit_reference(row, conn)
    for row in selected["working_dev"]
}

flagged_working = [
    row
    for row in selected["working_dev"]
    if working_audits[row["question_id"]]["concerns"]
]

for row in sorted(flagged_working, key=lambda item: item["question_id"]):
    audit = working_audits[row["question_id"]]
    result_preview = conn.execute(row["SQL"]).fetchmany(10)

    print("=" * 80)
    print("Question ID:", row["question_id"])
    print("Difficulty:", row["difficulty"])
    print("Question:", row["question"])
    print("Reference SQL:", row["SQL"])
    print("Result preview:", result_preview)
    print("Concern:", audit["concerns"])

Question ID: 5
Difficulty: medium
Question: How many male customers who are living in North Bohemia have average salary greater than 8000?
Reference SQL: SELECT COUNT(t1.client_id) FROM client AS t1 INNER JOIN district AS t2 ON t1.district_id = t2.district_id WHERE t1.gender = 'm' AND t2.a3 = 'north bohemia' AND t2.a11 > 8000
Result preview: [(0,)]
Concern: ['single numeric zero']
Question ID: 11
Difficulty: medium
Question: Among the accounts who have loan validity more than 12 months, list out the accounts that have the highest approved amount and have account opening date in 1993.
Reference SQL: SELECT t1.account_id FROM loan AS t1 INNER JOIN account AS t2 ON t1.account_id = t2.account_id WHERE STRFTIME('%y', t2.date) = '1993' AND t1.duration > 12 ORDER BY t1.amount DESC LIMIT 1
Result preview: []
Concern: ['empty result']
Question ID: 14
Difficulty: easy
Question: State different accounts who have account opening date before 1997 and own an amount of money greater than 3000USD
Refe

In [6]:
diagnostics = {
    "client.gender": """
        SELECT gender, COUNT(*)
        FROM client
        GROUP BY gender
        ORDER BY gender
    """,
    "loan.status": """
        SELECT status, COUNT(*)
        FROM loan
        GROUP BY status
        ORDER BY status
    """,
    "disp.type": """
        SELECT type, COUNT(*)
        FROM disp
        GROUP BY type
        ORDER BY type
    """,
    "account.frequency": """
        SELECT frequency, COUNT(*)
        FROM account
        GROUP BY frequency
        ORDER BY frequency
    """,
    "North Bohemia spelling": """
        SELECT DISTINCT A3
        FROM district
        WHERE LOWER(A3) = 'north bohemia'
    """,
    "account date behavior": """
        SELECT
            date,
            TYPEOF(date),
            STRFTIME('%Y', date),
            STRFTIME('%y', date)
        FROM account
        LIMIT 5
    """,
    "loan date behavior": """
        SELECT
            date,
            TYPEOF(date),
            STRFTIME('%Y', date),
            STRFTIME('%y', date)
        FROM loan
        LIMIT 5
    """,
}

for label, sql in diagnostics.items():
    print(f"\n{label}:")
    print(conn.execute(sql).fetchall())


client.gender:
[('F', 2645), ('M', 2724)]

loan.status:
[('A', 203), ('B', 31), ('C', 403), ('D', 45)]

disp.type:
[('DISPONENT', 869), ('OWNER', 4500)]

account.frequency:
[('POPLATEK MESICNE', 4167), ('POPLATEK PO OBRATU', 93), ('POPLATEK TYDNE', 240)]

North Bohemia spelling:
[('north Bohemia',)]

account date behavior:
[('1995-03-24', 'text', '1995', None), ('1993-02-26', 'text', '1993', None), ('1997-07-07', 'text', '1997', None), ('1996-02-21', 'text', '1996', None), ('1997-05-30', 'text', '1997', None)]

loan date behavior:
[('1994-01-05', 'text', '1994', None), ('1996-04-29', 'text', '1996', None), ('1997-12-08', 'text', '1997', None), ('1998-10-14', 'text', '1998', None), ('1998-04-19', 'text', '1998', None)]


In [7]:
import re

def make_reference_compatible(sql: str) -> tuple[str, list[str]]:
    compatible = sql
    applied_rules = []

    # FINCH uses a four-digit year literal but SQLite does not support %y.
    updated = re.sub(
        r"STRFTIME\(\s*'%y'\s*,",
        "STRFTIME('%Y',",
        compatible,
        flags=re.IGNORECASE,
    )
    if updated != compatible:
        applied_rules.append("sqlite_four_digit_year")
        compatible = updated

    # FINCH lowercases categorical literals while the database retains casing.
    updated = re.sub(
        r"\b([A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)?)"
        r"\s*=\s*('(?:''|[^'])*')",
        r"\1 = \2 COLLATE NOCASE",
        compatible,
    )
    if updated != compatible:
        applied_rules.append("case_insensitive_text_equality")
        compatible = updated

    return compatible, applied_rules

In [9]:
for row in sorted(flagged_working, key=lambda item: item["question_id"]):
    compatible_sql, applied_rules = make_reference_compatible(row["SQL"])
    result = conn.execute(compatible_sql).fetchmany(10)

    print("=" * 80)
    print("Question ID:", row["question_id"])
    print("Applied rules:", applied_rules)
    print("Compatible result:", result)

Question ID: 5
Applied rules: ['case_insensitive_text_equality']
Compatible result: [(280,)]
Question ID: 11
Applied rules: ['sqlite_four_digit_year']
Compatible result: [(10451,)]
Question ID: 14
Applied rules: ['sqlite_four_digit_year']
Compatible result: [(1,), (2,), (4,), (6,), (7,), (8,), (9,), (10,), (11,), (14,)]
Question ID: 49
Applied rules: ['case_insensitive_text_equality']
Compatible result: [(47,)]
Question ID: 81
Applied rules: ['sqlite_four_digit_year', 'case_insensitive_text_equality']
Compatible result: [(25.300191222790616,)]
Question ID: 99
Applied rules: ['case_insensitive_text_equality']
Compatible result: [(240,)]
Question ID: 105
Applied rules: ['case_insensitive_text_equality']
Compatible result: [(1, 18, 'Pisek'), (2, 1, 'Hl.m. Praha'), (4, 5, 'Kolin'), (6, 12, 'Pribram'), (7, 15, 'Cesky Krumlov'), (8, 51, 'Trutnov'), (9, 60, 'Prostejov'), (10, 57, 'Hodonin'), (12, 70, 'Karvina'), (13, 54, 'Brno - mesto')]


In [10]:
def audit_compatible_reference(
    row: dict,
    conn: sqlite3.Connection,
) -> dict:
    compatible_sql, applied_rules = make_reference_compatible(row["SQL"])

    try:
        cursor = conn.execute(compatible_sql)
        result = cursor.fetchall()
        column_count = len(cursor.description or [])

        concerns = []

        if len(result) == 0:
            concerns.append("empty result")

        if any(value is None for result_row in result for value in result_row):
            concerns.append("contains NULL")

        if (
            len(result) == 1
            and len(result[0]) == 1
            and isinstance(result[0][0], (int, float))
            and result[0][0] == 0
        ):
            concerns.append("single numeric zero")

        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": True,
            "row_count": len(result),
            "column_count": column_count,
            "applied_rules": applied_rules,
            "concerns": concerns,
        }

    except sqlite3.Error as error:
        return {
            "question_id": row["question_id"],
            "difficulty": row["difficulty"],
            "executes": False,
            "row_count": None,
            "column_count": None,
            "applied_rules": applied_rules,
            "concerns": [f"{type(error).__name__}: {error}"],
        }

working_compatible_audits = [
    audit_compatible_reference(row, conn)
    for row in selected["working_dev"]
]

print(
    "Executed:",
    sum(audit["executes"] for audit in working_compatible_audits),
    "/",
    len(working_compatible_audits),
)

print(
    "Still flagged:",
    [
        audit
        for audit in working_compatible_audits
        if not audit["executes"] or audit["concerns"]
    ],
)

Executed: 12 / 12
Still flagged: []


In [11]:
frozen_compatible_audits = [
    audit_compatible_reference(row, conn)
    for row in selected["frozen_eval"]
]

frozen_flagged = [
    audit
    for audit in frozen_compatible_audits
    if not audit["executes"] or audit["concerns"]
]

print(
    "Executed:",
    sum(audit["executes"] for audit in frozen_compatible_audits),
    "/",
    len(frozen_compatible_audits),
)
print("Still flagged:", frozen_flagged)

print(
    "Compatibility-rule usage:",
    dict(
        Counter(
            rule
            for audit in frozen_compatible_audits
            for rule in audit["applied_rules"]
        )
    ),
)

Executed: 24 / 24
Still flagged: [{'question_id': 104, 'difficulty': 'medium', 'executes': True, 'row_count': 1, 'column_count': 1, 'applied_rules': ['case_insensitive_text_equality'], 'concerns': ['contains NULL']}, {'question_id': 37, 'difficulty': 'hard', 'executes': True, 'row_count': 45, 'column_count': 1, 'applied_rules': ['case_insensitive_text_equality'], 'concerns': ['contains NULL']}]
Compatibility-rule usage: {'sqlite_four_digit_year': 6, 'case_insensitive_text_equality': 21}


In [12]:
all_compatible_audits = [
    audit_compatible_reference(row, conn)
    for row in eligible
]

all_flagged = [
    audit
    for audit in all_compatible_audits
    if not audit["executes"] or audit["concerns"]
]

print(
    "All eligible executed:",
    sum(audit["executes"] for audit in all_compatible_audits),
    "/",
    len(all_compatible_audits),
)
print("All eligible still flagged:", len(all_flagged))

print(
    "Remaining concern types:",
    dict(
        Counter(
            concern
            for audit in all_flagged
            for concern in audit["concerns"]
        )
    ),
)

print(
    "Compatibility-rule usage across eligible pool:",
    dict(
        Counter(
            rule
            for audit in all_compatible_audits
            for rule in audit["applied_rules"]
        )
    ),
)

All eligible executed: 103 / 103
All eligible still flagged: 3
Remaining concern types: {'contains NULL': 2, 'empty result': 1}
Compatibility-rule usage across eligible pool: {'case_insensitive_text_equality': 75, 'sqlite_four_digit_year': 24}


In [13]:
audit_by_id = {
    audit["question_id"]: audit
    for audit in all_compatible_audits
}

def is_clean_reference(question_id: int) -> bool:
    audit = audit_by_id[question_id]
    return audit["executes"] and not audit["concerns"]

working_ids = {
    row["question_id"]
    for row in selected["working_dev"]
}

replacement_frozen = []

for difficulty, required_count in split_spec["frozen_eval"].items():
    candidates = [
        row
        for row in by_difficulty[difficulty]
        if row["question_id"] not in working_ids
        and is_clean_reference(row["question_id"])
    ]

    replacement_frozen.extend(candidates[:required_count])

old_frozen_ids = {
    row["question_id"]
    for row in selected["frozen_eval"]
}
new_frozen_ids = {
    row["question_id"]
    for row in replacement_frozen
}

print("Removed:", sorted(old_frozen_ids - new_frozen_ids))
print("Added:", sorted(new_frozen_ids - old_frozen_ids))
print("Final frozen IDs:", sorted(new_frozen_ids))
print(
    "Final difficulty counts:",
    dict(Counter(row["difficulty"] for row in replacement_frozen)),
)

selected["frozen_eval"] = replacement_frozen

Removed: [37, 104]
Added: [31, 85]
Final frozen IDs: [4, 6, 10, 12, 13, 17, 28, 29, 30, 31, 35, 38, 39, 41, 47, 52, 53, 58, 62, 80, 82, 85, 93, 96]
Final difficulty counts: {'easy': 12, 'medium': 9, 'hard': 3}


In [14]:
final_frozen_audits = [
    audit_compatible_reference(row, conn)
    for row in selected["frozen_eval"]
]

print("Total:", len(final_frozen_audits))
print(
    "Executed:",
    sum(audit["executes"] for audit in final_frozen_audits),
)
print(
    "Flagged:",
    [
        audit
        for audit in final_frozen_audits
        if not audit["executes"] or audit["concerns"]
    ],
)

assert len(final_frozen_audits) == 24
assert all(audit["executes"] for audit in final_frozen_audits)
assert all(not audit["concerns"] for audit in final_frozen_audits)

Total: 24
Executed: 24
Flagged: []


In [15]:
manifest = {
    "manifest_version": 1,
    "benchmark": "domain-adapt-financial-v1",
    "created_on": "2026-09-25",
    "source": {
        "dataset_repository": "domyn/FINCH",
        "dataset_file": "finch_dataset.json",
        "dataset_sha256": file_sha256(dataset_path),
        "database_file": "text2sql-db/text2sql/bird/financial.sqlite",
        "database_sha256": file_sha256(db_path),
        "license": "CC-BY-NC-4.0",
    },
    "scope": {
        "db_name": "bird",
        "db_id": "financial",
        "upstream_partition": "dev",
    },
    "selection": {
        "algorithm": (
            "SHA-256 rank within difficulty using "
            "'{salt}:{difficulty}:{question_id}'"
        ),
        "salt": SELECTION_SALT,
        "orientation_exclusions": sorted(ORIENTATION_IDS),
        "ambiguous_reference_exclusions": sorted(
            audit["question_id"]
            for audit in all_flagged
        ),
        "working_dev": {
            "ids": sorted(
                row["question_id"]
                for row in selected["working_dev"]
            ),
            "difficulty_counts": dict(
                Counter(
                    row["difficulty"]
                    for row in selected["working_dev"]
                )
            ),
        },
        "frozen_eval": {
            "ids": sorted(
                row["question_id"]
                for row in selected["frozen_eval"]
            ),
            "difficulty_counts": dict(
                Counter(
                    row["difficulty"]
                    for row in selected["frozen_eval"]
                )
            ),
        },
    },
    "reference_compatibility": {
        "scope": "reference_sql_only",
        "rules": [
            {
                "id": "sqlite_four_digit_year",
                "description": (
                    "Replace unsupported SQLite STRFTIME('%y', ...) "
                    "with STRFTIME('%Y', ...)."
                ),
            },
            {
                "id": "case_insensitive_text_equality",
                "description": (
                    "Apply COLLATE NOCASE to reference-SQL equality "
                    "comparisons against string literals because FINCH "
                    "lowercases literals while the database retains casing."
                ),
            },
        ],
    },
    "reference_audit": {
        "requirements": [
            "SQL executes successfully",
            "result contains at least one row",
            "result contains no NULL values",
            "result is not a singleton numeric zero",
        ],
    },
    "leakage_policy": {
        "working_dev": (
            "May be inspected for evaluator, prompt, and harness development; "
            "must not be used as few-shot or training data."
        ),
        "frozen_eval": (
            "Questions, SQL, and results must remain uninspected during "
            "system development and must never be used for prompting, "
            "few-shot selection, synthetic generation, or training."
        ),
        "financial_database": (
            "All 106 FINCH BIRD financial examples are excluded from "
            "few-shot pools and model-training data."
        ),
    },
}

manifest_path = Path("../configs/eval_manifest_v1.json")
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8",
)

print(manifest_path.resolve())
print(manifest_path.read_text(encoding="utf-8"))

C:\Users\shchitt\Downloads\Projects\domain-adapt\configs\eval_manifest_v1.json
{
  "manifest_version": 1,
  "benchmark": "domain-adapt-financial-v1",
  "created_on": "2026-09-25",
  "source": {
    "dataset_repository": "domyn/FINCH",
    "dataset_file": "finch_dataset.json",
    "dataset_sha256": "c1e462743e4891fecf0fc10dbc6b0eb554fac56a186bdeedec9ccd662e0a4130",
    "database_file": "text2sql-db/text2sql/bird/financial.sqlite",
    "database_sha256": "d15d89cdb068a202b6f2b99342af44dffc1d52545b39ceaf62efdc0ba570101e",
    "license": "CC-BY-NC-4.0"
  },
  "scope": {
    "db_name": "bird",
    "db_id": "financial",
    "upstream_partition": "dev"
  },
  "selection": {
    "algorithm": "SHA-256 rank within difficulty using '{salt}:{difficulty}:{question_id}'",
    "salt": "domain-adapt-financial-v1",
    "orientation_exclusions": [
      1,
      2,
      3
    ],
    "ambiguous_reference_exclusions": [
      37,
      43,
      104
    ],
    "working_dev": {
      "ids": [
        5,
 

In [16]:
loaded_manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

assert loaded_manifest["source"]["dataset_sha256"] == (
    "c1e462743e4891fecf0fc10dbc6b0eb554fac56a186bdeedec9ccd662e0a4130"
)
assert loaded_manifest["source"]["database_sha256"] == (
    "d15d89cdb068a202b6f2b99342af44dffc1d52545b39ceaf62efdc0ba570101e"
)
assert len(loaded_manifest["selection"]["working_dev"]["ids"]) == 12
assert len(loaded_manifest["selection"]["frozen_eval"]["ids"]) == 24
assert set(
    loaded_manifest["selection"]["working_dev"]["ids"]
).isdisjoint(
    loaded_manifest["selection"]["frozen_eval"]["ids"]
)

print("Frozen manifest validated.")

Frozen manifest validated.
